<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####Working with dates
1. [Date functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#date-and-timestamp-functions)

####1. Requirement
You are given a dataframe as below

In [0]:
from pyspark.sql.functions import concat_ws

data_list = [(2022, 5, 18) , (9999, 12, 31), (-9999, 1, 1), (10000, 1, 1), 
             (-10000, 1, 1), (193, 5, 25),   (99, 5, 25),   (1000, 2, 29)]

df = (spark.createDataFrame(data_list).toDF("Y", "M", "D")
     .withColumn("date_str", concat_ws("-", "Y", "M", "D")))

df.display()

Y,M,D,date_str
2022,5,18,2022-5-18
9999,12,31,9999-12-31
-9999,1,1,-9999-1-1
10000,1,1,10000-1-1
-10000,1,1,-10000-1-1
193,5,25,193-5-25
99,5,25,99-5-25
1000,2,29,1000-2-29


####2. Convert strings to date

1. Spark validates the date against the Proleptic Gregorian calendar.
2. The negative years are BC, and the positive values are AD in Gregorian calander.
3. Valid dates are taken, and invalid dates throw an exception or taken as null

In [0]:
from pyspark.sql.functions import expr
df = (
    df.withColumn("valid_date", expr("try_to_date(date_str, 'y-M-d')"))
        .drop("Y", "M", "D")
)

df.display()

date_str,valid_date
2022-5-18,2022-05-18
9999-12-31,9999-12-31
-9999-1-1,-9999-01-01
10000-1-1,+10000-01-01
-10000-1-1,-10000-01-01
193-5-25,0193-05-25
99-5-25,0099-05-25
1000-2-29,null


####3. Add, Subtract days and months to date


In [0]:
from pyspark.sql.functions import date_add, date_sub, add_months, date_diff

df = (
    df.withColumns({
        "add_5_days": date_add("valid_date", 5),
        "sub_5_days": date_sub("valid_date", 5),
        "add_5_months": add_months("valid_date", 5),
        "sub_5_months": add_months("valid_date", -5)
    })
)

df.display()

date_str,valid_date,add_5_days,sub_5_days,add_5_months,sub_5_months
2022-5-18,2022-05-18,2022-05-23,2022-05-13,2022-10-18,2021-12-18
9999-12-31,9999-12-31,+10000-01-05,9999-12-26,+10000-05-31,9999-07-31
-9999-1-1,-9999-01-01,-9999-01-06,-10000-12-27,-9999-06-01,-10000-08-01
10000-1-1,+10000-01-01,+10000-01-06,9999-12-27,+10000-06-01,9999-08-01
-10000-1-1,-10000-01-01,-10000-01-06,-10001-12-27,-10000-06-01,-10001-08-01
193-5-25,0193-05-25,0193-05-30,0193-05-20,0193-10-25,0192-12-25
99-5-25,0099-05-25,0099-05-30,0099-05-20,0099-10-25,0098-12-25
1000-2-29,null,null,null,null,null


####4. Current date, date difference, and interval

In [0]:
from pyspark.sql.functions import current_date, date_diff, col
df = (
    df.withColumns({
        "current_date": current_date(),
        "delta_date_days": date_diff("add_5_months", "valid_date"),
        "delta_date_interval": col("current_date") - col("valid_date")
    })
)

df.display()

date_str,valid_date,add_5_days,sub_5_days,add_5_months,sub_5_months,current_date,delta_date_days,delta_date_interval
2022-5-18,2022-05-18,2022-05-23,2022-05-13,2022-10-18,2021-12-18,2026-05-11,153,INTERVAL '1454' DAY
9999-12-31,9999-12-31,+10000-01-05,9999-12-26,+10000-05-31,9999-07-31,2026-05-11,152,INTERVAL '-2912312' DAY
-9999-1-1,-9999-01-01,-9999-01-06,-10000-12-27,-9999-06-01,-10000-08-01,2026-05-11,151,INTERVAL '4392171' DAY
10000-1-1,+10000-01-01,+10000-01-06,9999-12-27,+10000-06-01,9999-08-01,2026-05-11,152,INTERVAL '-2912313' DAY
-10000-1-1,-10000-01-01,-10000-01-06,-10001-12-27,-10000-06-01,-10001-08-01,2026-05-11,152,INTERVAL '4392537' DAY
193-5-25,0193-05-25,0193-05-30,0193-05-20,0193-10-25,0192-12-25,2026-05-11,153,INTERVAL '669475' DAY
99-5-25,0099-05-25,0099-05-30,0099-05-20,0099-10-25,0098-12-25,2026-05-11,153,INTERVAL '703808' DAY
1000-2-29,null,null,null,null,null,2026-05-11,null,null


####5. Format date

In [0]:
from pyspark.sql.functions import date_format

df = (
    df.withColumn("fmt_date", date_format("valid_date", "dd MMM yyyy"))
)

df.display()

date_str,valid_date,add_5_days,sub_5_days,add_5_months,sub_5_months,current_date,delta_date_days,delta_date_interval,fmt_date
2022-5-18,2022-05-18,2022-05-23,2022-05-13,2022-10-18,2021-12-18,2026-05-11,153,INTERVAL '1454' DAY,18 May 2022
9999-12-31,9999-12-31,+10000-01-05,9999-12-26,+10000-05-31,9999-07-31,2026-05-11,152,INTERVAL '-2912312' DAY,31 Dec 9999
-9999-1-1,-9999-01-01,-9999-01-06,-10000-12-27,-9999-06-01,-10000-08-01,2026-05-11,151,INTERVAL '4392171' DAY,01 Jan -9999
10000-1-1,+10000-01-01,+10000-01-06,9999-12-27,+10000-06-01,9999-08-01,2026-05-11,152,INTERVAL '-2912313' DAY,01 Jan +10000
-10000-1-1,-10000-01-01,-10000-01-06,-10001-12-27,-10000-06-01,-10001-08-01,2026-05-11,152,INTERVAL '4392537' DAY,01 Jan -10000
193-5-25,0193-05-25,0193-05-30,0193-05-20,0193-10-25,0192-12-25,2026-05-11,153,INTERVAL '669475' DAY,25 May 0193
99-5-25,0099-05-25,0099-05-30,0099-05-20,0099-10-25,0098-12-25,2026-05-11,153,INTERVAL '703808' DAY,25 May 0099
1000-2-29,null,null,null,null,null,2026-05-11,null,null,null


&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>